In [3]:
import requests
import re
import json

def inspect_tweet_metadata(url: str):
    # 1. Extract Tweet ID
    twitter_re = re.compile(
        r"https?://(?:www\.)?(?:twitter\.com|x\.com)/\S+/status/(\d+)",
        re.IGNORECASE
    )
    match = twitter_re.search(url)
    
    if not match:
        print("❌ Invalid Twitter/X URL.")
        return
        
    tweet_id = match.group(1)
    print(f"✅ Extracted Tweet ID: {tweet_id}")
    print(f"Fetching data from Twitter Syndication API...\n")
    
    # 2. Construct Syndication URL
    synd_url = f"https://cdn.syndication.twimg.com/tweet-result?id={tweet_id}&lang=en&token=x"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        "Referer": "https://platform.twitter.com/",
    }
    
    # 3. Fetch Data
    try:
        response = requests.get(synd_url, headers=headers, timeout=10)
        
        if response.status_code == 404:
            print("❌ Error 404: Tweet not found (may be deleted or private).")
            return
        elif response.status_code != 200:
            print(f"❌ Error fetching data. Status Code: {response.status_code}")
            return
            
        data = response.json()
        
        # 4. Print the entire raw JSON payload
        print("================ RAW API RESPONSE ================")
        print(json.dumps(data, indent=4))
        print("==================================================")
        
        # 5. Print a quick summary of the keys your UI is looking for
        print("\n--- Quick Sanity Check ---")
        print(f"Likes (favorite_count): {data.get('favorite_count')}")
        print(f"Replies (conversation_count): {data.get('conversation_count')}")
        print(f"Quotes (quote_count): {data.get('quote_count')}")
        print(f"Mentions (entities.user_mentions): {len(data.get('entities', {}).get('user_mentions', []))}")
        
    except Exception as e:
        print(f"❌ Request failed: {e}")

# Replace this with your test tweet URL
test_url = "https://x.com/nytimes/status/2046242341845914078" 
inspect_tweet_metadata(test_url)

✅ Extracted Tweet ID: 2046242341845914078
Fetching data from Twitter Syndication API...

================ RAW API RESPONSE ================
{
    "__typename": "Tweet",
    "lang": "en",
    "favorite_count": 141,
    "possibly_sensitive": false,
    "created_at": "2026-04-20T14:59:26.000Z",
    "display_text_range": [
        0,
        184
    ],
    "entities": {
        "hashtags": [],
        "urls": [
            {
                "display_url": "nyti.ms/48HgBlk",
                "expanded_url": "https://nyti.ms/48HgBlk",
                "indices": [
                    161,
                    184
                ],
                "url": "https://t.co/2A8MFmdQEf"
            }
        ],
        "user_mentions": [],
        "symbols": []
    },
    "id_str": "2046242341845914078",
    "text": "Breaking News: The Supreme Court will consider whether Catholic preschools in Colorado that decline to enroll families with LGBTQ parents can get state funding. https://t.co/2A8MFmdQEf",


In [5]:
import requests
import re
import json

def inspect_syndication_api(url: str):
    twitter_re = re.compile(
        r"https?://(?:www\.)?(?:twitter\.com|x\.com)/\S+/status/(\d+)",
        re.IGNORECASE
    )
    match = twitter_re.search(url)
    
    if not match:
        print("❌ Invalid Twitter/X URL.")
        return
        
    tweet_id = match.group(1)
    print(f"✅ Extracted Tweet ID: {tweet_id}")
    
    synd_url = f"https://cdn.syndication.twimg.com/tweet-result?id={tweet_id}&lang=en&token=x"
    headers = {"User-Agent": "Mozilla/5.0"}
    
    try:
        response = requests.get(synd_url, headers=headers, timeout=10)
        if response.status_code != 200:
            print(f"❌ Error fetching data. Status Code: {response.status_code}")
            return
            
        data = response.json()
        
        print("================ RAW API RESPONSE ================")
        print(json.dumps(data, indent=4))
        print("==================================================")
        
        # Aggressive Hunt for Metrics
        print("\n--- 📊 Metrics Found in JSON ---")
        
        # Likes
        print(f"❤️ Likes (favorite_count): {data.get('favorite_count', 'MISSING')}")
        
        # Replies (Checking both known keys)
        replies = data.get('conversation_count', data.get('reply_count', 'MISSING'))
        print(f"💬 Replies (conversation_count): {replies}")
        
        # Reposts / Retweets
        print(f"🔁 Reposts (retweet_count): {data.get('retweet_count', 'MISSING')}")
        
        # Quotes
        print(f"🗣️ Quotes (quote_count): {data.get('quote_count', 'MISSING')}")
        
        # Views (Usually stored inside a nested 'ext_views' dictionary)
        views = data.get('ext_views', {}).get('count', 'MISSING')
        print(f"👁️ Views (ext_views.count): {views}")
        
        # Bookmarks (Checking if they added it recently)
        print(f"🔖 Bookmarks: {data.get('bookmark_count', 'MISSING')}")
        
    except Exception as e:
        print(f"❌ Request failed: {e}")

# ---------------------------------------------------------
# BONUS: How to parse the aria-label if you use a local web scraper
# ---------------------------------------------------------
def parse_aria_label(aria_text: str):
    print("\n--- 🕵️‍♂️ HTML Aria-Label Extraction ---")
    
    metrics = {
        "replies": r"([\d,]+)\s+replies",
        "reposts": r"([\d,]+)\s+reposts",
        "likes": r"([\d,]+)\s+likes",
        "bookmarks": r"([\d,]+)\s+bookmarks",
        "views": r"([\d,]+)\s+views"
    }
    
    results = {}
    for key, pattern in metrics.items():
        match = re.search(pattern, aria_text, re.IGNORECASE)
        if match:
            # Remove commas and convert to integer
            results[key] = int(match.group(1).replace(",", ""))
        else:
            results[key] = None
            
    for k, v in results.items():
        print(f"{k.capitalize()}: {v}")


# --- RUN THE SCRIPT ---
test_url = "https://x.com/nytimes/status/2046242341845914078" 
inspect_syndication_api(test_url)

# Test the aria-label parser using your exact example
sample_aria = "50 replies, 63 reposts, 141 likes, 13 bookmarks, 80694 views"
parse_aria_label(sample_aria)

✅ Extracted Tweet ID: 2046242341845914078
================ RAW API RESPONSE ================
{
    "__typename": "Tweet",
    "lang": "en",
    "favorite_count": 141,
    "possibly_sensitive": false,
    "created_at": "2026-04-20T14:59:26.000Z",
    "display_text_range": [
        0,
        184
    ],
    "entities": {
        "hashtags": [],
        "urls": [
            {
                "display_url": "nyti.ms/48HgBlk",
                "expanded_url": "https://nyti.ms/48HgBlk",
                "indices": [
                    161,
                    184
                ],
                "url": "https://t.co/2A8MFmdQEf"
            }
        ],
        "user_mentions": [],
        "symbols": []
    },
    "id_str": "2046242341845914078",
    "text": "Breaking News: The Supreme Court will consider whether Catholic preschools in Colorado that decline to enroll families with LGBTQ parents can get state funding. https://t.co/2A8MFmdQEf",
    "user": {
        "id_str": "807095",
     

In [9]:
import requests
import re

def audit_twitter_features(url: str):
    # 1. Extract ID
    match = re.search(r"status/(\d+)", url, re.IGNORECASE)
    if not match:
        print("❌ Invalid URL")
        return
    
    tweet_id = match.group(1)
    synd_url = f"https://cdn.syndication.twimg.com/tweet-result?id={tweet_id}&lang=en&token=x"
    headers = {"User-Agent": "Mozilla/5.0"}
    
    # 2. Fetch Data
    resp = requests.get(synd_url, headers=headers)
    if resp.status_code != 200:
        print(f"❌ Failed to fetch. Status: {resp.status_code}")
        return
        
    data = resp.json()
    entities = data.get("entities", {})
    tweet_text = data.get("text", "")
    
    print(f"✅ Successfully fetched tweet: '{tweet_text[:50]}...'\n")
    print("================ FEATURE AUDIT ================\n")

    # --- 3. THE AT-RISK METADATA COLUMNS ---
    print("🚨 API METADATA FEATURES (Check these closely):")
    
    # Favourites (Likes)
    favs = data.get('favorite_count')
    print(f"   [ ] 'favourites' -> Found in API? {'✅ Yes' if favs is not None else '❌ No'} (Value: {favs})")
    
    # Replies (using conversation_count)
    replies = data.get('conversation_count')
    print(f"   [ ] 'replies'    -> Found in API? {'✅ Yes' if replies is not None else '❌ No'} (Value: {replies})")
    
    # Quotes
    quotes = data.get('quote_count')
    print(f"   [ ] 'quotes'     -> Found in API? {'✅ Yes' if quotes is not None else '❌ No'} (Value: {quotes})")
    
    # Mentions
    mentions_list = entities.get('user_mentions', [])
    has_mentions = len(mentions_list) > 0
    print(f"   [ ] 'mentions'   -> Provided by API? {'✅ Yes' if has_mentions else '⚠️ Empty/Missing'} (Count: {len(mentions_list)})")
    
    # Hashtags
    hashtags_list = entities.get('hashtags', [])
    has_tags = len(hashtags_list) > 0
    print(f"   [ ] 'hashtags'   -> Provided by API? {'✅ Yes' if has_tags else '⚠️ Empty/Missing'} (Count: {len(hashtags_list)})")
    
    # URLs
    urls_list = entities.get('urls', [])
    has_urls = len(urls_list) > 0
    print(f"   [ ] 'URLs'       -> Provided by API? {'✅ Yes' if has_urls else '⚠️ Empty/Missing'} (Count: {len(urls_list)})\n")


    # --- 4. THE SAFE LEXICAL COLUMNS ---
    print("🟢 TEXT-DERIVED FEATURES (100% Safe):")
    print("   Because the API returned the 'tweet_text', you can safely calculate ALL of these:")
    
    safe_cols = [
        'unique_count', 'total_count', 'Word count', 'Max word length',
        'Min word length', 'Average word length', 'present_verbs', 'past_verbs',
        'adjectives', 'adverbs', 'adpositions', 'pronouns', 'TOs',
        'determiners', 'conjunctions', 'dots', 'exclamation', 'questions',
        'ampersand', 'capitals', 'digits', 'long_word_freq', 'short_word_freq'
    ]
    
    # Print them in a clean grid
    for i in range(0, len(safe_cols), 3):
        row = safe_cols[i:i+3]
        print("   " + ", ".join(row))


# Test with a tweet you KNOW has mentions, quotes, and hashtags
test_urls = ["https://x.com/nytimes/status/2046242341845914078", "https://x.com/TheDunkCentral/status/2046297158924881964", "https://x.com/USMarshalsHQ/status/2046239558707073259"]  
for test_url in test_urls:
    print(audit_twitter_features(test_url))

✅ Successfully fetched tweet: 'Breaking News: The Supreme Court will consider whe...'

================ FEATURE AUDIT ================

🚨 API METADATA FEATURES (Check these closely):
   [ ] 'favourites' -> Found in API? ✅ Yes (Value: 143)
   [ ] 'replies'    -> Found in API? ✅ Yes (Value: 50)
   [ ] 'quotes'     -> Found in API? ❌ No (Value: None)
   [ ] 'mentions'   -> Provided by API? ⚠️ Empty/Missing (Count: 0)
   [ ] 'hashtags'   -> Provided by API? ⚠️ Empty/Missing (Count: 0)
   [ ] 'URLs'       -> Provided by API? ✅ Yes (Count: 1)

🟢 TEXT-DERIVED FEATURES (100% Safe):
   Because the API returned the 'tweet_text', you can safely calculate ALL of these:
   unique_count, total_count, Word count
   Max word length, Min word length, Average word length
   present_verbs, past_verbs, adjectives
   adverbs, adpositions, pronouns
   TOs, determiners, conjunctions
   dots, exclamation, questions
   ampersand, capitals, digits
   long_word_freq, short_word_freq
None
✅ Successfully fetched t

In [10]:
import requests
import re

def comprehensive_twitter_audit(url: str):
    # 1. Extract ID
    match = re.search(r"status/(\d+)", url, re.IGNORECASE)
    if not match:
        print("❌ Invalid URL")
        return
    
    tweet_id = match.group(1)
    synd_url = f"https://cdn.syndication.twimg.com/tweet-result?id={tweet_id}&lang=en&token=x"
    headers = {"User-Agent": "Mozilla/5.0"}
    
    # 2. Fetch Data
    resp = requests.get(synd_url, headers=headers)
    if resp.status_code != 200:
        print(f"❌ Failed to fetch. Status: {resp.status_code}")
        return
        
    data = resp.json()
    user = data.get("user", {})
    entities = data.get("entities", {})
    tweet_text = data.get("text", "")
    
    print(f"✅ Successfully fetched tweet: '{tweet_text[:60]}...'\n")
    print("================ FULL FEATURE AUDIT ================\n")

    # --- BLOCK 1: USER METADATA (The riskiest block) ---
    print("👤 USER PROFILE FEATURES:")
    user_keys = [
        "followers_count", 
        "friends_count", 
        "favourites_count", 
        "statuses_count", 
        "listed_count", 
        "following"
    ]
    
    if not user:
        print("   ❌ CRITICAL: The entire 'user' dictionary is MISSING from the API response.")
    else:
        for key in user_keys:
            val = user.get(key)
            status = "✅ Yes" if val is not None else "❌ No "
            print(f"   [ ] '{key:<18}' -> Found? {status} (Value: {val})")


    # --- BLOCK 2: ENGAGEMENT METADATA ---
    print("\n📊 TWEET ENGAGEMENT FEATURES:")
    
    retweets = data.get('retweet_count')
    print(f"   [ ] 'retweets'         -> Found? {'✅ Yes' if retweets is not None else '❌ No '} (Value: {retweets})")
    
    likes = data.get('favorite_count')
    print(f"   [ ] 'favourites'(Likes)-> Found? {'✅ Yes' if likes is not None else '❌ No '} (Value: {likes})")
    
    replies = data.get('conversation_count')
    print(f"   [ ] 'replies'          -> Found? {'✅ Yes' if replies is not None else '❌ No '} (Value: {replies})")
    
    quotes = data.get('quote_count')
    print(f"   [ ] 'quotes'           -> Found? {'✅ Yes' if quotes is not None else '❌ No '} (Value: {quotes})")


    # --- BLOCK 3: ENTITY METADATA ---
    print("\n🔗 TWEET ENTITY FEATURES:")
    
    mentions = entities.get('user_mentions', [])
    print(f"   [ ] 'mentions'         -> Array Present? {'✅ Yes' if 'user_mentions' in entities else '❌ No '} (Count: {len(mentions)})")
    
    hashtags = entities.get('hashtags', [])
    print(f"   [ ] 'hashtags'         -> Array Present? {'✅ Yes' if 'hashtags' in entities else '❌ No '} (Count: {len(hashtags)})")
    
    urls = entities.get('urls', [])
    print(f"   [ ] 'URLs'             -> Array Present? {'✅ Yes' if 'urls' in entities else '❌ No '} (Count: {len(urls)})\n")

    print("====================================================")


# Test with a URL
test_urls = ["https://x.com/nytimes/status/2046242341845914078", "https://x.com/TheDunkCentral/status/2046297158924881964", "https://x.com/USMarshalsHQ/status/2046239558707073259"]  
for test_url in test_urls:
    print(comprehensive_twitter_audit(test_url))

✅ Successfully fetched tweet: 'Breaking News: The Supreme Court will consider whether Catho...'

================ FULL FEATURE AUDIT ================

👤 USER PROFILE FEATURES:
   [ ] 'followers_count   ' -> Found? ❌ No  (Value: None)
   [ ] 'friends_count     ' -> Found? ❌ No  (Value: None)
   [ ] 'favourites_count  ' -> Found? ❌ No  (Value: None)
   [ ] 'statuses_count    ' -> Found? ❌ No  (Value: None)
   [ ] 'listed_count      ' -> Found? ❌ No  (Value: None)
   [ ] 'following         ' -> Found? ❌ No  (Value: None)

📊 TWEET ENGAGEMENT FEATURES:
   [ ] 'retweets'         -> Found? ❌ No  (Value: None)
   [ ] 'favourites'(Likes)-> Found? ✅ Yes (Value: 144)
   [ ] 'replies'          -> Found? ✅ Yes (Value: 50)
   [ ] 'quotes'           -> Found? ❌ No  (Value: None)

🔗 TWEET ENTITY FEATURES:
   [ ] 'mentions'         -> Array Present? ✅ Yes (Count: 0)
   [ ] 'hashtags'         -> Array Present? ✅ Yes (Count: 0)
   [ ] 'URLs'             -> Array Present? ✅ Yes (Count: 1)

None
✅ Successf